# Reachy Voice Assistant — guided walkthrough

A step-by-step runbook for the all-AWS voice robot: **Amazon Transcribe (STT) → Amazon Bedrock (LLM) → Amazon Polly (TTS)**, built on pipecat and driving a Reachy Mini.

```
mic → LocalAudioTransport.input → Transcribe(STT) → user aggregator (Silero VAD)
    → Bedrock(LLM) → Polly(TTS) → transport.output → ReachyDriver → speaker/robot
```

**How to use this notebook.** The cells below run every *checkable* step: config, credentials, a Bedrock reply, a Polly voice sample, and the unit tests. The final **live conversation loop cannot run inside a notebook cell** — it is a long-running, real-time audio pipeline that owns your microphone and speaker — so the last step hands off to a terminal command.

> Select the project's `.venv` as the notebook kernel (Python 3.12).

## Step 0 — Prerequisites

Run these once in a terminal from `use-cases/reachy-voice-assistant/` (see `README.md`):

```bash
brew install portaudio                 # macOS: PyAudio needs it
python3.12 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt        # full app runtime
pip install -r requirements-dev.txt    # tests + this notebook (jupyterlab, boto3)
```

You also need **AWS credentials** (SSO/profile/env) and **Bedrock model access** enabled for your model in your region. The cells below check both.

## Step 1 — Environment and resolved configuration

In [ ]:
import sys, os
sys.path.insert(0, 'src')   # so 'voicebot' imports without an editable install
from voicebot.config import load_config

cfg = load_config('config.yaml')
print('Python :', sys.version.split()[0])
print('Region :', cfg.region)
print('Model  :', cfg.llm.model_id)
print('Voice  :', cfg.tts.voice, '(', cfg.tts.engine, 'engine )')
print('Robot  :', cfg.robot.backend)

## Step 2 — AWS credentials

Confirms the default credential chain resolves. If it fails, run `aws sso login` (or your org's login) in a terminal, then re-run this cell.

In [ ]:
import boto3
from botocore.exceptions import BotoCoreError, ClientError

try:
    ident = boto3.client('sts', region_name=cfg.region).get_caller_identity()
    print('Authenticated as:', ident['Arn'])
except (BotoCoreError, ClientError) as exc:
    print('No usable AWS credentials yet:\n ', exc)
    print('\nRun `aws sso login` in a terminal, then re-run this cell.')

## Step 3 — Bedrock LLM (the brain)

Sends one prompt to your configured model via the Bedrock `converse` API. This also confirms **model access** is enabled. An `AccessDeniedException` means you must enable the model under **Bedrock console → Model access** for your region.

In [ ]:
brt = boto3.client('bedrock-runtime', region_name=cfg.region)

# Build inference config, omitting temperature when it is null (some models,
# e.g. reasoning models like Grok, reject the temperature field).
inference = {'maxTokens': 100}
if cfg.llm.temperature is not None:
    inference['temperature'] = cfg.llm.temperature

try:
    resp = brt.converse(
        modelId=cfg.llm.model_id,
        system=[{'text': 'You are Polly, a friendly robot. Reply in one short sentence.'}],
        messages=[{'role': 'user', 'content': [{'text': 'Say hello and introduce yourself.'}]}],
        inferenceConfig=inference,
    )
    # Reasoning models return a reasoningContent block first; pick the text block.
    blocks = resp['output']['message']['content']
    text = next((b['text'] for b in blocks if 'text' in b), '(no text block returned)')
    print('Model reply:', text)
    print('Latency (ms):', resp.get('metrics', {}).get('latencyMs'))
except ClientError as exc:
    err = exc.response['Error']
    print('Bedrock call failed:', err['Code'], '-', err['Message'])
    print('\nAccessDenied  -> enable model access for', cfg.llm.model_id, 'in region', cfg.region)
    print('Validation about temperature -> set llm.temperature: null in config.yaml')

## Step 4 — Amazon Polly (the voice)

Synthesizes a sample with your configured Polly voice/engine and plays the saved MP3 through the same local playback API used by the Python examples. This is the same voice the live assistant will speak with.

In [ ]:
from playsound3 import playsound

polly = boto3.client('polly', region_name=cfg.region)
try:
    r = polly.synthesize_speech(
        Text="Hello! I'm Polly, your Reachy voice assistant.",
        VoiceId=cfg.tts.voice,
        Engine=cfg.tts.engine,
        OutputFormat='mp3',
    )
    audio = r['AudioStream'].read()
    with open('polly_sample.mp3', 'wb') as fh:
        fh.write(audio)
    print('Synthesized', len(audio), 'bytes with voice', cfg.tts.voice)
    playsound('polly_sample.mp3')
except ClientError as exc:
    err = exc.response['Error']
    print('Polly call failed:', err['Code'], '-', err['Message'])

## Step 5 — Amazon Transcribe (the ears)

Transcribe here runs as a **real-time streaming** service inside the pipecat pipeline (`AWSTranscribeSTTService`), consuming your microphone live — there is no simple one-shot cell demo for streaming STT. You exercise it directly in the live run (Step 7): speak, and watch the transcript drive the model. The preflight below confirms the service imports and your credentials/region are wired.

In [ ]:
import subprocess, sys, os
env = {**os.environ, 'PYTHONPATH': 'src'}
res = subprocess.run([sys.executable, 'app.py', '--check'], env=env,
                     capture_output=True, text=True)
# Filter the noisy pipecat banner lines for readability.
for line in res.stdout.splitlines():
    if 'Pipecat' not in line:
        print(line)

## Step 6 — Run the unit tests

Config parsing, the FakeReachy robot double, and the ReachyDriver frame-mapping logic. No AWS, robot, or audio hardware required.

In [ ]:
import subprocess, sys, os
env = {**os.environ, 'PYTHONPATH': 'src'}
res = subprocess.run([sys.executable, '-m', 'pytest', 'tests/', '-q'], env=env,
                     capture_output=True, text=True)
print(res.stdout[-2000:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-1000:])

## Step 7 — Run the live assistant (in a terminal)

The real-time voice loop owns the mic/speaker and blocks until you stop it, so run it in a **terminal**, not here.

**First, a no-hardware smoke test** — validates the full AWS speech loop and your mic/speaker with a fake robot:

```bash
source .venv/bin/activate
python app.py --robot fake
```

It should greet you out loud, then reply when you speak. Press `Ctrl-C` to stop.

**Then bring in the robot** — a daemon in one terminal, the app in another. Sim vs. real is decided by *which daemon runs*; the `app.py` command is identical.

**Real robot — Reachy Mini Lite** (USB). Plain `python`, no `mjpython`, no `--sim`:

```bash
# terminal 1 — USB robot daemon
reachy-mini-daemon           # add -p /dev/cu.usbmodem<serial> if auto-detect fails

# terminal 2
source .venv/bin/activate
python app.py --robot reachy
```

**Simulator** (no hardware):

```bash
# terminal 1 — MuJoCo simulator daemon
mjpython -m reachy_mini.daemon.app.main --sim --scene minimal

# terminal 2
source .venv/bin/activate
python app.py --robot reachy
```

Watch for: a nod + `welcoming1` move on start, a *listening* pose while you talk, and a *speaking* pose while it replies.

On the Lite, the robot's mic array and speaker are one USB card ("Reachy Mini Audio"), which the app finds by name — replies always come out of the robot. If the array transcribes you poorly, or the bot starts hearing itself, set `audio.mic: laptop` in `config.yaml` and speak into a host mic held away from the robot.

## Troubleshooting & cleanup

- **`AccessDeniedException` (Bedrock):** enable model access in the Bedrock console for your region.
- **No audio / device errors:** confirm PyAudio installed (`brew install portaudio` first). The app logs the devices it resolved at startup (`Audio devices: mic=… speaker …`); pin `audio.input_device_index` / `output_device_index` in `config.yaml` if it picked the wrong card. On a real robot the daemon also holds the robot's sound card — `reachy-mini-daemon --no-media` frees it.
- **Robot mic doesn't hear you well, or the bot talks to itself:** set `audio.mic: laptop` in `config.yaml` to listen on the host's default input instead; the speaker stays the robot's.
- **Robot not found (real robot):** the daemon finds the motor bus over USB by VID:PID `1a86:55d3`; pass `-p /dev/cu.usbmodem<serial>` if auto-detection fails.
- **`mjpython: command not found`:** you're likely in a `uv` env — use a plain `venv` (see README). Only the simulator needs `mjpython`.
- **Costs:** Transcribe (per-minute), Polly (per-character), Bedrock (per-token) bill while running.
- **Cleanup:** stop the app and daemon with `Ctrl-C`; delete `polly_sample.mp3`; no persistent AWS resources are created.